In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

model_name = "lukasjanek/slovakbert-skquad-qa"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Pipeline approach
# qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# print("\n--- QA Example (Pipeline Way) ---")

# result_pipeline_1 = qa_pipeline(question="Kde leží Slovensko?", context="Slovensko je krajina ležiaca v strednej Európe. Hlavným mestom Slovenska je Bratislava.")
# print(f"\nQuestion: '{result_pipeline_1['question']}'")
# print(f"Answer: '{result_pipeline_1['answer']}' (score: {result_pipeline_1['score']:.4f})")

# result_pipeline_2 = qa_pipeline(question="Kto bol Milan Rastislav Štefánik?", context="Milan Rastislav Štefánik bol slovenský astronóm, generál a politik. Bol jedným zo zakladateľov Československa.")
# print(f"\nQuestion: '{result_pipeline_2['question']}'")
# print(f"Answer: '{result_pipeline_2['answer']}' (score: {result_pipeline_2['score']:.4f})")

print("\n--- Manual QA Example ---")

def get_manual_qa(question, context, model, tokenizer, device, max_length=512, top_k=1):
    # 1. Tokenize question + context together as a pair
    #    The model expects [CLS] question [SEP] context [SEP]
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_offsets_mapping=False,  # we will use token-level span and decode directly
    ).to(device)

    # 2. Perform model inference — outputs start_logits and end_logits over all tokens
    with torch.no_grad():
        outputs = model(**inputs)
        start_logits = outputs.start_logits  # [1, seq_len]
        end_logits = outputs.end_logits      # [1, seq_len]

    input_ids = inputs["input_ids"][0]
    sequence_ids = inputs.sequence_ids(0)  # 0 = question tokens, 1 = context tokens, None = special

    context_mask = torch.tensor(
        [1 if s == 1 else 0 for s in sequence_ids],
        dtype=torch.float,
        device=device,
    )

    masked_start = start_logits[0] + (context_mask - 1) * 1e9
    masked_end   = end_logits[0]   + (context_mask - 1) * 1e9

    start_probs = torch.softmax(masked_start, dim=-1)
    end_probs   = torch.softmax(masked_end,   dim=-1)

    top_k_starts = torch.topk(start_probs, top_k)
    top_k_ends   = torch.topk(end_probs,   top_k)

    best_score = -1
    best_start = 0
    best_end   = 0

    for start_idx, start_score in zip(top_k_starts.indices, top_k_starts.values):
        for end_idx, end_score in zip(top_k_ends.indices, top_k_ends.values):
            if end_idx >= start_idx:  # span must be non-empty and forward
                combined = (start_score * end_score).item()
                if combined > best_score:
                    best_score = combined
                    best_start = start_idx.item()
                    best_end   = end_idx.item()

    # 4. Decode the winning span back to a string
    answer_ids = input_ids[best_start : best_end + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    return {
        "question": question,
        "context": context,
        "answer": answer,
        "score": best_score,
        "span": (best_start, best_end),
    }


question_1 = "Kde leží Slovensko?"
context_1  = "Slovensko je krajina ležiaca v strednej Európe. Hlavným mestom Slovenska je Bratislava, ktorá leží na rieke Dunaj."
result_1 = get_manual_qa(question_1, context_1, model, tokenizer, device)
print(f"\nQuestion: '{result_1['question']}'")
print(f"Context:  '{result_1['context']}'")
print(f"Answer:   '{result_1['answer']}' (score: {result_1['score']:.4f})")


question_2 = "Čím bol Milan Rastislav Štefánik?"
context_2  = "Milan Rastislav Štefánik bol slovenský astronóm, generál a politik. Bol jedným zo zakladateľov Československa a zahynul pri leteckom nešťastí v roku 1919."
result_2 = get_manual_qa(question_2, context_2, model, tokenizer, device)
print(f"\nQuestion: '{result_2['question']}'")
print(f"Context:  '{result_2['context']}'")
print(f"Answer:   '{result_2['answer']}' (score: {result_2['score']:.4f})")

question_3 = "Koľko percent HDP tvorí rozpočtový deficit?"
context_3  = "Slovenská vláda plánuje v budúcom roku znížiť rozpočtový deficit verejných financií na 3,8 percenta hrubého domáceho produktu. Cieľom je dosiahnutie vyrovnaného rozpočtu do roku 2028."
result_3 = get_manual_qa(question_3, context_3, model, tokenizer, device)
print(f"\nQuestion: '{result_3['question']}'")
print(f"Context:  '{result_3['context']}'")
print(f"Answer:   '{result_3['answer']}' (score: {result_3['score']:.4f})")

Loading lukasjanek/slovakbert-skquad-qa...


config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- Manual QA Example ---

Question: 'Kde leží Slovensko?'
Context:  'Slovensko je krajina ležiaca v strednej Európe. Hlavným mestom Slovenska je Bratislava, ktorá leží na rieke Dunaj.'
Answer:   'Slovensko je krajina ležiaca v strednej Európe' (score: 0.4914)

Question: 'Čím bol Milan Rastislav Štefánik?'
Context:  'Milan Rastislav Štefánik bol slovenský astronóm, generál a politik. Bol jedným zo zakladateľov Československa a zahynul pri leteckom nešťastí v roku 1919.'
Answer:   ' bol slovenský astronóm, generál a politik' (score: 0.3436)

Question: 'Koľko percent HDP tvorí rozpočtový deficit?'
Context:  'Slovenská vláda plánuje v budúcom roku znížiť rozpočtový deficit verejných financií na 3,8 percenta hrubého domáceho produktu. Cieľom je dosiahnutie vyrovnaného rozpočtu do roku 2028.'
Answer:   ' 3,8 percenta hrubého domáceho produktu' (score: 0.4464)


In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")